In [1]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine('postgresql://localhost/steam_analytics')
df = pd.read_sql('SELECT * FROM reviews LIMIT 10000', engine)
print('Loaded rows:', len(df))
df.head()

Loaded rows: 10000


,id,review_text,hours_played,helpful,funny,is_recommended,date,game_name,username
0,1,The game itself is also super fun. The PvP and...,39.9,1152,13,True,14 September,"Warhammer 40,000: Space Marine 2",Sentinowl\n224 products in account
1,2,Never cared much about Warhammer until this ga...,91.5,712,116,True,13 September,"Warhammer 40,000: Space Marine 2",userpig\n248 products in account
2,3,A salute to all the fallen battle brothers who...,43.3,492,33,True,14 September,"Warhammer 40,000: Space Marine 2",Imparat0r\n112 products in account
3,4,this game feels like it was made in the mid 20...,16.8,661,15,True,14 September,"Warhammer 40,000: Space Marine 2",Fattest_falcon
4,5,Reminds me of something I've lost. A genuine g...,24.0,557,4,True,12 September,"Warhammer 40,000: Space Marine 2",Jek\n410 products in account


In [2]:
# How many positive vs negative reviews?
print(df['is_recommended'].value_counts())
print()

# What percentage are positive?
positive_pct = df['is_recommended'].mean() * 100
print(f'Positive reviews: {positive_pct:.1f}%')
print(f'Negative reviews: {100-positive_pct:.1f}%')

is_recommended
True     8223
False    1777
Name: count, dtype: int64

Positive reviews: 82.2%
Negative reviews: 17.8%


In [3]:
# Which games have the most reviews?
df['game_name'].value_counts().head(10)

game_name
Warhammer 40,000: Space Marine 2    3400
Black Myth: Wukong                  3367
Counter-Strike 2                    3207
Once Human                            14
ELDEN RING                            10
Party Animals                          2
Name: count, dtype: int64

In [4]:
# Check full dataset game distribution
df_games = pd.read_sql("""
    SELECT game_name, COUNT(*) as review_count
    FROM reviews
    GROUP BY game_name
    ORDER BY review_count DESC
    LIMIT 20
""", engine)

df_games

,game_name,review_count
0,Golf With Your Friends,5010
1,Hollow Knight,5010
2,eFootball™,5010
3,FINAL FANTASY XIV Online,5010
4,Caves of Qud,5010
5,Grand Theft Auto V,5010
6,Aimlabs,5010
7,Cities: Skylines II,5010
8,Fallout: New Vegas,5010
9,Dying Light,5010


In [5]:
# Average hours played for positive vs negative reviews
df_hours = pd.read_sql("""
    SELECT 
        is_recommended,
        ROUND(AVG(hours_played)::numeric, 2) as avg_hours,
        COUNT(*) as review_count
    FROM reviews
    WHERE hours_played IS NOT NULL
    GROUP BY is_recommended
""", engine)

df_hours

,is_recommended,avg_hours,review_count
0,False,115.19,172034
1,True,131.79,745918


In [6]:
# Find potentially sarcastic reviews - high hours but negative
df_sarcastic = pd.read_sql("""
    SELECT game_name, review_text, hours_played, is_recommended
    FROM reviews
    WHERE is_recommended = False
    AND hours_played > 500
    ORDER BY hours_played DESC
    LIMIT 5
""", engine)

for i, row in df_sarcastic.iterrows():
    print(f"Game: {row['game_name']}")
    print(f"Hours: {row['hours_played']}")
    print(f"Review: {row['review_text'][:200]}")
    print("---")


Game: NBA 2K24
Hours: 999.9
Review: 2023 Mike Wang doesn't care about PC players.
---
Game: War Thunder
Hours: 999.8
Review: 2013 This update 1.37 is so F**** Shi*
---
Game: EA SPORTS FC™ 24
Hours: 999.8
Review: 2023 Game has good graphics, animations, its realistic... Game itself is broken, servers are awfulit takes 20-40 seconds just to open the store or to scroll to another slide. Gameplay is very rigged/b
---
Game: EA SPORTS FC™ 24
Hours: 999.8
Review: unfathomably ass game
---
Game: ARK: Survival Ascended
Hours: 999.7
Review: 2023 Early Access Review NOPE, don't do it even if you love Ark like me and my friends did, it will only disappoint you. Same copy and paste coding, with a team that obviously has not learned from pas
---
